# EX6 ANN MNIST

In [ ]:
import torchvision
import torch
import torch.nn as nn
import torch.optim as opt
from matplotlib import pyplot as plt
import numpy as np
from torchvision.transforms import transforms

## data preparation

In [ ]:
train_dataset = torchvision.datasets.MNIST("../DATA/",download=True,train=True,transform=transforms.ToTensor())
test_dataset = torchvision.datasets.MNIST("../DATA/",download=True,train=False,transform=transforms.ToTensor())


let's visualize

In [ ]:
plt.imshow(train_dataset.data[0],cmap='gray')
plt.title(str(train_dataset.targets[0].item()))


data processing continued

In [ ]:
print(f'train x shape {train_dataset.data.shape} y_train {train_dataset.targets.shape}\ntest x shape {test_dataset.data.shape} y_test {test_dataset.targets.shape}')

In [ ]:
train_loader = torch.utils.data.DataLoader(dataset=train_dataset,batch_size=128,shuffle=True)
test_loader = torch.utils.data.DataLoader(dataset=test_dataset,batch_size=128,shuffle=False)

## build the model

In [ ]:
model = nn.Sequential(
    nn.Linear(28*28,128),
    nn.ReLU(),
    nn.Linear(128,10)
    # nn.Softmax() - not needed 
)

## set device to GPU(if exists)

In [ ]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(device)
model.to(device=device)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = opt.Adam(model.parameters(),lr=0.0001)

## Training the model

In [ ]:
EPOCHS = 10
train_losses = []
test_losses = []
train_accuracy = []
test_accuracy = []

for epoch in range(EPOCHS): 
    n_correct = 0
    n_total = 0
    train_loss = []
    # 0. batch loop 
    for inputs,targets in train_loader:
        #1. zero gradients  that optimizer are in charge
        optimizer.zero_grad()
        
        # put inputs and targets into device(GPU)
        inputs,targets = inputs.to(device),targets.to(device)

        # flatten the input (because it's ANN NxD)
        inputs = inputs.view(-1,28*28)

        outputs = model(inputs)

        loss = criterion(outputs,targets)
        loss.backward()
        optimizer.step()
        _,indxs = torch.max(outputs,1)
        
        #calculate accuracy
        with torch.no_grad():
            n_total += indxs.shape[0]
            n_correct +=np.sum(indxs.cpu().numpy() == targets.cpu().numpy())
        #batch loss
        train_loss.append(loss.item())

    train_losses.append(np.mean(train_loss))
    train_accuracy.append(n_correct/n_total)
    test_loss = []
    #test
    n_correct = 0
    n_total = 0
    for inputs,targets in test_loader:
        inputs,targets = inputs.to(device),targets.to(device)
        inputs = inputs.view(-1,28*28)

        outputs = model(inputs)
        loss = criterion(outputs,targets)
        test_loss.append(loss.item())
        _,indxs = torch.max(outputs,1)

    #calculate accuracy
        with torch.no_grad():
            n_total += indxs.shape[0]
            n_correct +=np.sum(indxs.cpu().numpy() == targets.cpu().numpy())
    test_losses.append(np.mean(test_loss))
    test_accuracy.append(n_correct/n_total)


    print(f'train_loss {train_losses[-1]} test_loss {test_losses[-1]} train acc: {train_accuracy[-1]} test acc: {test_accuracy[-1]}')
    #acc = n_correct /n_total

In [ ]:
#predict:

# _,predictions = torch.max(outputs,1)

In [ ]:
x = [i for i in range(EPOCHS)]
plt.plot(x,train_losses,label='train_loss',color='orange')
plt.plot(x,test_losses,label='test_loss',color='red')
plt.legend()

In [ ]:
x = [i for i in range(EPOCHS)]
plt.plot(x,train_accuracy,label='train_acc',color='blue')
plt.plot(x,test_accuracy,label='test_acc',color='green')
plt.legend()